# Assignment 1:  Disease (Heart Disease) Prediction Using Machine Learning
# Name: Tasnuba Tasnim
# ID: 2671508


Dataset: Kaggle Heart Disease Dataset  
`johnsmith88/heart-disease-dataset`

Target:

0 = No Heart Disease

1 = Heart Disease


## Phase 1(ID:2671508) — Data Engineering & Feature Preparation

### Includes
Data Preparation: Download the Kaggle heart disease dataset, Load the CSV into a Pandas DataFrame, ensuring target variables and feature names are explicitly mapped.

Data Integrity Check:Programmatically verify the presence of missing or null values across all dimensions, implementing a conditional log statement, Check duplicates, Encode categorical variables if required, Define X = input features and y = Heart Disease

Feature Scaling: Perform feature selection, Split into training and testing sets, Apply StandardScaler to normalize the selected feature subset to ensure zero mean and unit variance.



In [ ]:
# Phase 1(ID:2671508): Data Engineering & Feature Preparation

!pip -q install kagglehub

import os
import glob
import numpy as np
import pandas as pd
import kagglehub

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", None)

# Download and load dataset
path = kagglehub.dataset_download(
    "johnsmith88/heart-disease-dataset"
)
print("Dataset Path:", path)

csv_files = glob.glob(
    os.path.join(path, "*.csv")
)
if len(csv_files) == 0:
    raise FileNotFoundError("No CSV file found.")

heart_files = [
    file for file in csv_files
    if os.path.basename(file).lower() == "heart.csv"
]
csv_path = heart_files[0] if heart_files else csv_files[0]

df = pd.read_csv(csv_path)

print("\nDataset loaded successfully.")

print("\nFirst 5 Rows:")
print(df.head().to_string(index=False))

print("\nDataset Shape:")
print(df.shape)

print("\nColumn Names:")
print(df.columns.tolist())

print("\nData Types:")
print(df.dtypes)

print("\nStatistical Summary:")
print(df.describe().T)

# Target information
print("\nTarget Counts:")
print(df["target"].value_counts().sort_index())

print("\nTarget Percentage:")
print(
    df["target"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)
print("\nTarget Meaning:")
print("0 = No Heart Disease")
print("1 = Heart Disease")

# Missing value check
missing_values = df.isnull().sum()

print("\nMissing Values:")
print(missing_values)

total_missing = df.isnull().sum().sum()

if total_missing == 0:
    print("\nLOG: No missing/null values found.")
else:
    print(f"\nLOG: {total_missing} missing values found.")

    numeric_columns = df.select_dtypes(
        include=np.number
    ).columns

    for column in numeric_columns:
        if df[column].isnull().any():
            df[column] = df[column].fillna(
                df[column].median()
            )
    print("LOG: Missing values filled using median.")
    print(
        "Remaining Missing Values:",
        df.isnull().sum().sum()
    )
# Duplicate check
duplicate_count = df.duplicated().sum()

print("\nDuplicate Rows:")
print(duplicate_count)

if duplicate_count > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    print("LOG: Duplicate rows removed.")
else:
    print("LOG: No duplicate rows found.")

print("\nDataset Shape After Cleaning:")
print(df.shape)

# Encode categorical features
categorical_columns = [
    "cp",
    "restecg",
    "slope",
    "thal"
]

df_encoded = pd.get_dummies(
    df,
    columns=categorical_columns,
    dtype=int
)
print("\nDataset Shape After Encoding:")
print(df_encoded.shape)

print("\nFirst 5 Rows After Encoding:")
print(df_encoded.head().to_string(index=False))

# Define features and target
X = df_encoded.drop(
    columns=["target"]
)
y = df_encoded["target"]

print("\nX Shape:")
print(X.shape)

print("\ny Shape:")
print(y.shape)

print("\nAll Features Used:")

for number, feature in enumerate(
    X.columns,
    start=1
):
    print(f"{number}. {feature}")

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)
print("\nTraining and Testing Data:")

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

# Use all features
selected_features = X.columns.tolist()

print("\nTotal Features Used:")
print(len(selected_features))

# Feature scaling
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(
    X_train
)
X_test_scaled = scaler.transform(
    X_test
)
print("\nFeature Scaling Completed.")
print("\nScaled Training Feature Mean:")
print(
    np.round(
        X_train_scaled.mean(axis=0),
        4
    )
)
print("\nScaled Training Feature Standard Deviation:")
print(
    np.round(
        X_train_scaled.std(axis=0),
        4
    )
)
print("\nPhase 1 Completed Successfully.")

## Phase 2(ID:2671508): Model Implementation

### Includes
Support Vector Machine (SVM);

Logistic Regression



In [ ]:
# Phase 2 (ID: 2671508): Model Implementation

from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression

# Train SVM
svm_model = SVC(
    kernel="rbf",
    C=1.0,
    gamma="scale",
    probability=True,
    random_state=42
)

svm_model.fit(X_train_scaled, y_train)

svm_pred = svm_model.predict(X_test_scaled)
svm_prob = svm_model.predict_proba(X_test_scaled)[:, 1]

print("SVM model trained successfully.")
print("First 20 SVM Predictions:")
print(svm_pred[:20])


# Train Logistic Regression
lr_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

lr_model.fit(X_train_scaled, y_train)

lr_pred = lr_model.predict(X_test_scaled)
lr_prob = lr_model.predict_proba(X_test_scaled)[:, 1]

print("\nLogistic Regression model trained successfully.")
print("First 20 Logistic Regression Predictions:")
print(lr_pred[:20])


# Compare actual and predicted values
comparison_prediction = pd.DataFrame({
    "Actual": y_test.values,
    "SVM Prediction": svm_pred,
    "Logistic Regression Prediction": lr_pred
})

print("\nPrediction Comparison:")
print(
    comparison_prediction
    .head(20)
    .to_string(index=False)
)


# Prediction counts
print("\nSVM Prediction Counts:")
print(
    pd.Series(svm_pred)
    .value_counts()
    .sort_index()
)

print("\nLogistic Regression Prediction Counts:")
print(
    pd.Series(lr_pred)
    .value_counts()
    .sort_index()
)

print("\n0 = No Heart Disease")
print("1 = Heart Disease")

print("\nPHASE 2 COMPLETED SUCCESSFULLY")

## Phase 3(ID:2671508)Model Evaluation & Comparison

### Includes
For SVM : SVM Accuracy, Precision, Recall, F1-score; SVM TN, FP, FN, TP confusion matrix heatmap.

For Logistic Regression : Logistic Regression Accuracy, Precision, Recall, F1-score; Logistic Regression TN, FP, FN, TP confusion matrix heatmap.

ROC-AUC and ROC curve for both models; Final comparison table;Model comparison chart; Best model based on accuracy.

In [ ]:
# Phase 3(ID:2671508): Model Evaluation, Visualization & Comparison
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_curve,
    roc_auc_score
)

# SVM evaluation
svm_accuracy = accuracy_score(y_test, svm_pred)
svm_precision = precision_score(y_test, svm_pred, zero_division=0)
svm_recall = recall_score(y_test, svm_pred, zero_division=0)
svm_f1 = f1_score(y_test, svm_pred, zero_division=0)

print("SVM MODEL DETAILS")
print(f"Accuracy  : {svm_accuracy:.4f}")
print(f"Precision : {svm_precision:.4f}")
print(f"Recall    : {svm_recall:.4f}")
print(f"F1-score  : {svm_f1:.4f}")

svm_cm = confusion_matrix(y_test, svm_pred)
svm_tn, svm_fp, svm_fn, svm_tp = svm_cm.ravel()

print("\nSVM Confusion Matrix Details:")
print("True Negative  (TN):", svm_tn)
print("False Positive (FP):", svm_fp)
print("False Negative (FN):", svm_fn)
print("True Positive  (TP):", svm_tp)

svm_labels = [
    [f"TN\n{svm_tn}", f"FP\n{svm_fp}"],
    [f"FN\n{svm_fn}", f"TP\n{svm_tp}"]
]

plt.figure(figsize=(6, 5))

sns.heatmap(
    svm_cm,
    annot=svm_labels,
    fmt="",
    cmap="Blues",
    cbar=False,
    xticklabels=["No Disease", "Disease"],
    yticklabels=["No Disease", "Disease"]
)
plt.title("SVM Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.tight_layout()
plt.show()

# Logistic Regression evaluation
lr_accuracy = accuracy_score(y_test, lr_pred)
lr_precision = precision_score(y_test, lr_pred, zero_division=0)
lr_recall = recall_score(y_test, lr_pred, zero_division=0)
lr_f1 = f1_score(y_test, lr_pred, zero_division=0)

print("\nLOGISTIC REGRESSION MODEL DETAILS")
print(f"Accuracy  : {lr_accuracy:.4f}")
print(f"Precision : {lr_precision:.4f}")
print(f"Recall    : {lr_recall:.4f}")
print(f"F1-score  : {lr_f1:.4f}")

lr_cm = confusion_matrix(y_test, lr_pred)
lr_tn, lr_fp, lr_fn, lr_tp = lr_cm.ravel()

print("\nLogistic Regression Confusion Matrix Details:")
print("True Negative  (TN):", lr_tn)
print("False Positive (FP):", lr_fp)
print("False Negative (FN):", lr_fn)
print("True Positive  (TP):", lr_tp)

lr_labels = [
    [f"TN\n{lr_tn}", f"FP\n{lr_fp}"],
    [f"FN\n{lr_fn}", f"TP\n{lr_tp}"]
]
plt.figure(figsize=(6, 5))

sns.heatmap(
    lr_cm,
    annot=lr_labels,
    fmt="",
    cmap="Greens",
    cbar=False,
    xticklabels=["No Disease", "Disease"],
    yticklabels=["No Disease", "Disease"]
)
plt.title("Logistic Regression Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.tight_layout()
plt.show()

# ROC-AUC for both models
svm_auc = roc_auc_score(y_test, svm_prob)
lr_auc = roc_auc_score(y_test, lr_prob)

print("\nROC-AUC DETAILS")
print(f"SVM ROC-AUC                 : {svm_auc:.4f}")
print(f"Logistic Regression ROC-AUC : {lr_auc:.4f}")

svm_fpr, svm_tpr, _ = roc_curve(y_test, svm_prob)
lr_fpr, lr_tpr, _ = roc_curve(y_test, lr_prob)

plt.figure(figsize=(8, 6))

plt.plot(
    svm_fpr,
    svm_tpr,
    linewidth=2,
    label=f"SVM (AUC = {svm_auc:.3f})"
)

plt.plot(
    lr_fpr,
    lr_tpr,
    linewidth=2,
    label=f"Logistic Regression (AUC = {lr_auc:.3f})"
)
plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random Classifier"
)
plt.title("ROC Curve - SVM vs Logistic Regression")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Final comparison
comparison_df = pd.DataFrame({
    "Model": [
        "SVM",
        "Logistic Regression"
    ],
    "Accuracy": [
        svm_accuracy,
        lr_accuracy
    ],
    "Precision": [
        svm_precision,
        lr_precision
    ],
    "Recall": [
        svm_recall,
        lr_recall
    ],
    "F1-score": [
        svm_f1,
        lr_f1
    ],
    "ROC-AUC": [
        svm_auc,
        lr_auc
    ]
})
print("\nFINAL MODEL COMPARISON")
print(comparison_df.round(4).to_string(index=False))

comparison_chart = comparison_df.melt(
    id_vars="Model",
    var_name="Metric",
    value_name="Score"
)
plt.figure(figsize=(11, 6))

sns.barplot(
    data=comparison_chart,
    x="Metric",
    y="Score",
    hue="Model"
)
plt.title("Performance Comparison: SVM vs Logistic Regression")
plt.xlabel("Evaluation Metric")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.legend(title="Model")
plt.tight_layout()
plt.show()

print("\nFINAL RESULT")

print("\nSVM:")
print(f"Accuracy  : {svm_accuracy:.4f}")
print(f"Precision : {svm_precision:.4f}")
print(f"Recall    : {svm_recall:.4f}")
print(f"F1-score  : {svm_f1:.4f}")
print(f"ROC-AUC   : {svm_auc:.4f}")

print("\nLogistic Regression:")
print(f"Accuracy  : {lr_accuracy:.4f}")
print(f"Precision : {lr_precision:.4f}")
print(f"Recall    : {lr_recall:.4f}")
print(f"F1-score  : {lr_f1:.4f}")
print(f"ROC-AUC   : {lr_auc:.4f}")

if svm_accuracy > lr_accuracy:
    print("\nBest Model Based on Accuracy: SVM")
elif lr_accuracy > svm_accuracy:
    print("\nBest Model Based on Accuracy: Logistic Regression")
else:
    print("\nBoth Models Have the Same Accuracy")

print("\nPHASE 3 COMPLETED SUCCESSFULLY")


# -----------------------------------------------------------------------------------------------------------
# Create and save all plots
plot_folder = "result"
os.makedirs(plot_folder, exist_ok=True)


# SVM Confusion Matrix
svm_labels = [
    [f"TN\n{svm_tn}", f"FP\n{svm_fp}"],
    [f"FN\n{svm_fn}", f"TP\n{svm_tp}"]
]

plt.figure(figsize=(6, 5))

sns.heatmap(
    svm_cm,
    annot=svm_labels,
    fmt="",
    cmap="Blues",
    cbar=False,
    xticklabels=["No Disease", "Disease"],
    yticklabels=["No Disease", "Disease"]
)

plt.title("SVM Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.tight_layout()

plt.savefig(
    os.path.join(plot_folder, "svm_confusion_matrix.png"),
    dpi=300,
    bbox_inches="tight"
)

plt.close()


# Logistic Regression Confusion Matrix
lr_labels = [
    [f"TN\n{lr_tn}", f"FP\n{lr_fp}"],
    [f"FN\n{lr_fn}", f"TP\n{lr_tp}"]
]

plt.figure(figsize=(6, 5))

sns.heatmap(
    lr_cm,
    annot=lr_labels,
    fmt="",
    cmap="Greens",
    cbar=False,
    xticklabels=["No Disease", "Disease"],
    yticklabels=["No Disease", "Disease"]
)

plt.title("Logistic Regression Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.tight_layout()

plt.savefig(
    os.path.join(
        plot_folder,
        "logistic_regression_confusion_matrix.png"
    ),
    dpi=300,
    bbox_inches="tight"
)

plt.close()


# ROC Curve
plt.figure(figsize=(8, 6))

plt.plot(
    svm_fpr,
    svm_tpr,
    linewidth=2,
    label=f"SVM (AUC = {svm_auc:.3f})"
)

plt.plot(
    lr_fpr,
    lr_tpr,
    linewidth=2,
    label=f"Logistic Regression (AUC = {lr_auc:.3f})"
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random Classifier"
)

plt.title("ROC Curve - SVM vs Logistic Regression")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    os.path.join(
        plot_folder,
        "roc_curve_comparison.png"
    ),
    dpi=300,
    bbox_inches="tight"
)

plt.close()


# Model Comparison Chart
plt.figure(figsize=(11, 6))

sns.barplot(
    data=comparison_chart,
    x="Metric",
    y="Score",
    hue="Model"
)

plt.title("Performance Comparison: SVM vs Logistic Regression")
plt.xlabel("Evaluation Metric")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.legend(title="Model")
plt.tight_layout()

plt.savefig(
    os.path.join(
        plot_folder,
        "model_performance_comparison.png"
    ),
    dpi=300,
    bbox_inches="tight"
)

plt.close()
print("\nSaved Files:")
for file in os.listdir(plot_folder):
  print("-", file)
